# Origin Classification with TF-IDF

Baseline pipeline for country-level origin classification using tasting text.

In [17]:
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

In [18]:
DATA_PATH = 'Data/final_coffee_reviews.csv'
RANDOM_STATE = 42
MIN_SAMPLES_PER_CLASS = 10

df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
df.head(2)

Shape: (8889, 23)


,URL,all_text,Rating,Roaster,Coffee Name,Roaster Location,Coffee Origin,Roast Level,Agtron,Est. Price,...,Acidity/Structure,Body,Flavor,Aftertaste,With Milk,Blind Assessment,Notes,Who Should Drink It,Bottom Line,Combined_Acidity
0,https://www.coffeereview.com/review/100-arabic...,89\nCaffe Bomrad\n100% Arabica 100% Italiano\n...,89.0,Caffe Bomrad,100% Arabica 100% Italiano,"Torino, Italy",Not disclosed.,Medium,48/65,$54.00/1 Kilogram,...,NaN,8.0,8.0,7.0,8.0,evaluated as espresso. smoothly round aroma: t...,Roasted in Northern Italy and distributed in N...,A strong-charactered Northern Italian styled e...,NaN,NaN
1,https://www.coffeereview.com/review/100-arabic...,"87\nLucaff?\n100% Arabica, Black Label (ESE po...",87.0,Lucaff?,"100% Arabica, Black Label (ESE pod)","Padenghe sul Garda, Italy",Not disclosed.,Dark,0/80,NaN,...,NaN,8.0,7.0,7.0,8.0,produced from an ese pod on a francisfrancis! ...,ESE (Easy Serving Espresso) pods are wafer-lik...,An attractive pod espresso for big milk drinks.,NaN,NaN


In [22]:
df["origin_country"] = (
    df["Coffee Origin"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.rstrip(".")
    .str.split()
    .str[-1]          # take last word
    .replace("", pd.NA)
)

In [23]:
df["origin_country"] = df["origin_country"].str.title()


origin_country
Ethiopia     1702
Colombia      936
Kenya         702
Disclosed     608
Guatemala     581
NaN           564
Indonesia     460
Rica          395
Panama        348
Salvador      252
Brazil        247
Rwanda        180
Hawai’I       137
Nicaragua     137
Honduras      136
Peru          132
Hawaii        132
Mexico        120
Guinea         96
Burundi        89
Name: count, dtype: int64

In [20]:
import re
import pandas as pd

def extract_country(origin):
    if pd.isna(origin):
        return None

    s = str(origin).strip().strip(".")
    if s == "":
        return None

    # remove unusable labels
    if re.search(r"not disclosed|blend|various|multiple", s, flags=re.I):
        return None

    # split common separators, take last part (usually country)
    parts = [p.strip() for p in re.split(r"[,;/]", s) if p.strip()]
    if not parts:
        return None

    country = parts[-1]

    # optional normalization
    mapping = {
        "usa": "United States",
        "u.s.a": "United States",
        "u.s.a.": "United States",
        "hawaii": "United States",
    }
    return mapping.get(country.lower(), country.title())

df["origin_country"] = df["Coffee Origin"].apply(extract_country)


In [11]:
def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = text.lower().strip()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text


def extract_country(origin: str) -> str | None:
    if not isinstance(origin, str):
        return None
    t = origin.strip().strip('.')
    if t == '':
        return None
    if re.search(r'not disclosed|blend|various', t, flags=re.IGNORECASE):
        return None

    parts = [p.strip() for p in re.split(r'[,;/]', t) if p.strip()]
    if not parts:
        return None
    country = parts[-1]

    country_map = {
        'usa': 'United States',
        'u.s.a.': 'United States',
        'united states': 'United States',
        'hawaii': 'United States',
    }

    key = country.lower()
    return country_map.get(key, country.title())

In [12]:
# Build text input (tasting-focused)
df['text_main'] = df['Blind Assessment'].fillna('').map(normalize_text)

# Build country-level target
df['origin_country'] = df['Coffee Origin'].map(extract_country)

# Keep valid rows only
work = df[(df['text_main'].str.len() >= 30) & (df['origin_country'].notna())].copy()

# Remove tiny classes for a stable baseline
counts = work['origin_country'].value_counts()
valid_classes = counts[counts >= MIN_SAMPLES_PER_CLASS].index
work = work[work['origin_country'].isin(valid_classes)].copy()

print('Rows after filtering:', len(work))
print('Num classes:', work['origin_country'].nunique())
work['origin_country'].value_counts().head(15)

Rows after filtering: 7211
Num classes: 71


origin_country
Southern Ethiopia         1052
Colombia                   775
Guatemala                  493
Indonesia                  458
South-Central Kenya        399
Ethiopia                   359
Costa Rica                 339
South-Central Ethiopia     244
Brazil                     224
Western Panama             222
El Salvador                219
Kenya                      202
Rwanda                     157
Honduras                   132
Nicaragua                  122
Name: count, dtype: int64

In [13]:
X = work['text_main']
y = work['origin_country']

# 70/15/15 split (stratified)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

print('Train:', len(X_train), 'Val:', len(X_val), 'Test:', len(X_test))

Train: 5047 Val: 1082 Test: 1082


In [14]:
tfidf_lr = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.90,
        sublinear_tf=True
    )),
    ('clf', LogisticRegression(
        max_iter=3000,
        solver='saga',
        class_weight='balanced',
        n_jobs=-1,
        random_state=RANDOM_STATE
    ))
])

tfidf_lr.fit(X_train, y_train)

g:\Anaconda\Lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.9, min_df=3, ngram_range=(1, 2),
                                 sublinear_tf=True)),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=3000,
                                    n_jobs=-1, random_state=42,
                                    solver='saga'))])

In [15]:
def evaluate(model, X_data, y_true, split_name='split'):
    y_pred = model.predict(X_data)
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)

    print(f'== {split_name} ==')
    print(f'Accuracy:  {acc:.4f}')
    print(f'Precision: {p:.4f}')
    print(f'Recall:    {r:.4f}')
    print(f'F1-macro:  {f1:.4f}')

    return y_pred


_ = evaluate(tfidf_lr, X_val, y_val, 'Validation')
y_test_pred = evaluate(tfidf_lr, X_test, y_test, 'Test')

print('\nTop classes report (test):')
top_labels = y_test.value_counts().head(15).index
print(classification_report(y_test, y_test_pred, labels=top_labels, zero_division=0))

== Validation ==
Accuracy:  0.0194
Precision: 0.0563
Recall:    0.0306
F1-macro:  0.0249
== Test ==
Accuracy:  0.0102
Precision: 0.0351
Recall:    0.0395
F1-macro:  0.0109

Top classes report (test):
                        precision    recall  f1-score   support

     Southern Ethiopia       0.00      0.00      0.00       158
              Colombia       0.00      0.00      0.00       116
             Guatemala       0.00      0.00      0.00        74
             Indonesia       1.00      0.01      0.03        68
   South-Central Kenya       0.67      0.03      0.06        60
              Ethiopia       0.00      0.00      0.00        54
            Costa Rica       0.00      0.00      0.00        51
South-Central Ethiopia       0.00      0.00      0.00        37
        Western Panama       0.00      0.00      0.00        34
                Brazil       0.00      0.00      0.00        33
           El Salvador       0.00      0.00      0.00        33
                 Kenya       0.

In [16]:
# Optional: preview predictions
preview = pd.DataFrame({
    'text': X_test.head(5).values,
    'true_origin': y_test.head(5).values,
    'pred_origin': tfidf_lr.predict(X_test.head(5))
})
preview

,text,true_origin,pred_origin
0,"smoky-sweet, chocolaty. scorched mesquite, dar...",Haiti,Zambia
1,"the roast dominates the coffee, though gently....",South-Central Brazil,Northwestern Guatemala
2,"rich-toned, chocolaty and floral. baking choco...",“Big Island” Of Hawaii,Zambia
3,"floral, spice-toned. jasmine, pink peppercorn,...",Colombia,Yemen
4,evaluated as espresso. lushly but resonantly b...,South-Central Ethiopia,Zambia
